If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec23_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 23: Percentiles and the Bootstrap
v.ekc

Estimation begins: we have *one* sample and want to guess a population parameter. The tool is audacious — resample from your own sample — and it works. But first, percentiles. (Slides: KL Lecture 23 deck.)

**Today's sections:**
1. Percentiles
2. A population: SF city compensation
3. Estimating the parameter
4. The bootstrap
5. The middle 95%

In [ ]:
from datascience import *
%matplotlib inline
path_data = '../../../assets/data/'
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import numpy as np
import warnings
warnings.filterwarnings("ignore")

---
## 1. Percentiles

The p-th percentile: sort the data, take the value at least p% of the way through.

### 📋 Board Reference

| Code | What it does |
|---|---|
| `np.sort(x)` | sorted copy of the array |
| `percentile(55, x)` | the 55th percentile of `x` |
| rule | always an actual data value — no averaging |
| rank | first value ≥ p% of the way through the sorted list |

In [ ]:
# Manually compute the 55th percentile.
x = make_array(43, 20, 51, 7, 28, 34)

In [ ]:
# Step 1. Sort the data
np.sort(x)

In [ ]:
# Step 2. Figure out where 55th percentile would be.

In [ ]:
np.arange(1, 7)/6

In [ ]:
np.sort(x).item(3)

In [ ]:
# Alternatively: One line of code
percentile(55, x)

### Try It 1: True or False? 🤔

With `s = make_array(1, 3, 5, 7, 9)`, predict each **before** running:

1. `percentile(10, s) == 0`
2. `percentile(39, s) == percentile(40, s)`
3. `percentile(40, s) == percentile(41, s)`
4. `percentile(50, s) == 5`

In [ ]:
# Discussion question
s = make_array(1, 3, 5, 7, 9)

In [ ]:
percentile(10, s) == 0

In [ ]:
percentile(39, s) == percentile(40, s)

In [ ]:
percentile(40, s) == percentile(41, s)

In [ ]:
percentile(50, s) == 5

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*False, True, False, True. Percentiles are always actual data values (so #1 is False — it's 1), and small changes in p only matter when they cross a rank boundary (that's #2 vs. #3).*

</details>

---
## 2. A Population: San Francisco City Compensation 💰

Every SF city employee in 2019 — a full **population**, so for once we know the truth and can check ourselves. (We drop people below a part-time minimum wage salary first.)

In [ ]:
sf = Table.read_table('san_francisco_2019.csv')
sf.show(3)

In [ ]:
# Who made the most money
sf.sort('Total Compensation', descending=True).show(5)

In [ ]:
# Who made the least money
sf.sort('Total Compensation', descending=False).show(5)

In [ ]:
# $15/hr, 20 hr/wk, 50 weeks

min_salary = 15 * 20 * 50
sf = sf.where('Salary', are.above(min_salary))

In [ ]:
sf.num_rows

In [ ]:
sf_bins = np.arange(0, 726000, 25000)
sf.hist('Total Compensation', bins=sf_bins)

### The Parameter: Population Median

In [ ]:
pop_median = percentile(50, sf.column('Total Compensation'))
pop_median

---
## 3. Estimating the Parameter (Pretend It's Unknown)

In real life we'd only have a sample. Take one of size 400 — its median is an estimate, and every re-run gives a different one.

In [ ]:
our_sample = sf.sample(400, with_replacement=False)

In [ ]:
our_sample.hist('Total Compensation', bins=sf_bins)

In [ ]:
percentile(50, our_sample.column('Total Compensation') )

But in the real world we won't be able to keep going back to the population. How to generate a new random sample *without going back to the population?*

---
## 4. The Bootstrap 👢

The problem: quantifying our uncertainty seems to require many samples, but we can't go back to the population. The audacious fix (KL deck, slides 14–21): **the sample resembles the population — so resample from the sample.**

### 📋 Board Reference

| Bootstrap rule | Why |
|---|---|
| draw from the **original sample** | it's our best stand-in for the population |
| **with replacement** | otherwise you'd just get the same sample back |
| **same size** as the original | variability depends on sample size |
| `our_sample.sample()` | does all three by default! |

In [ ]:
# Default behavior of tbl.sample:
# at random with replacement,
# the same number of times as rows of tbl

bootstrap_sample = our_sample.sample()

In [ ]:
bootstrap_sample.hist('Total Compensation', bins=sf_bins)

## Bootstrap Sample Median
This is one estimate of the population median.

In [ ]:
percentile(50, bootstrap_sample.column('Total Compensation'))

In [ ]:
def one_bootstrap_median():
    # draw the bootstrap sample
    resample = our_sample.sample()
    # return the median total compensation in the bootstrap sample
    return percentile(50, resample.column('Total Compensation'))

In [ ]:
one_bootstrap_median()

In [ ]:
# Generate the medians of 1000 bootstrap samples
num_repetitions = 1000
bstrap_medians = make_array()
for i in np.arange(num_repetitions):
    bstrap_medians = np.append(bstrap_medians, one_bootstrap_median())

In [ ]:
resampled_medians = Table().with_column('Bootstrap Sample Median', bstrap_medians)
median_bins=np.arange(120000, 160000, 2000)
resampled_medians.hist(bins = median_bins)

# Plotting parameters; you can ignore this code
parameter_green = '#32CD32'
plots.ylim(-0.000005, 0.00014)
plots.scatter(pop_median, 0, color='red', s=40, zorder=2)
plots.title('Bootstrap Medians and the Parameter (Red Dot)');

---
## 5. The Middle 95% of Bootstrap Estimates

Cut off 2.5% on each side of the bootstrap medians — the interval that remains is our **95% confidence interval**.

In [ ]:
left = percentile(2.5, bstrap_medians)
right = percentile(97.5, bstrap_medians)

make_array(left, right)

In [ ]:
resampled_medians.hist(bins = median_bins)

# Plotting parameters; you can ignore this code
plots.ylim(-0.000005, 0.00014)
plots.plot(make_array(left, right), make_array(0, 0), color='yellow', lw=3, zorder=1)
plots.scatter(pop_median, 0, color=parameter_green, s=40, zorder=2);

> **Check the plot:** the red dot (the true population median — remember, we secretly know it) lands inside the yellow interval. Most samples produce an interval that catches the parameter; next lecture, we'll see exactly what "95% confident" does and doesn't mean.

---
> **Today's takeaway:** one sample can measure its own uncertainty — resample it, recompute the statistic, and take the middle 95%. Homework 8 runs on this.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — bootstrap CI
```python
def one_bootstrap_stat():
    return percentile(50, our_sample.sample().column('col'))

stats = make_array()
for i in np.arange(1000):
    stats = np.append(stats, one_bootstrap_stat())
make_array(percentile(2.5, stats), percentile(97.5, stats))
```